# 10 · Figuras finales (Entrega 3, fase A)

Conjunto único de figuras para el informe a partir de `results/08/` y `results/09/`. **No se recalcula ningún modelo**: solo lectura, agregación liviana y graficado. Todo lo reutilizable vive en `figuras.py`.

**Estilo único:** colores fijos por fuente (Okabe-Ito) y por macrocategoría (Tol *muted*, cada tópico hereda el de su macrotema); letra al tamaño final (16 cm de ancho); PDF vectorial y PNG a 300 dpi en `results/figuras/`. **Toda serie temporal** lleva la banda del cambio de archivo de El Tiempo (2006) y una banda más tenue en 2000 (régimen de breves y codificación). Los cambios de agenda se dibujan en el inicio del nuevo régimen (año − 0,5).

**Excepciones declaradas:** F11b y F12 usan tablas que el 09 calculó y no exportó. Se re-derivan con el mismo código, configuración y semillas del 09 y se verifican contra `resumen_09.json`. Si no coinciden, la figura no se exporta y el índice remite a la figura del 09.

**Celdas costosas (⏱):** F06 (bootstrap de 7 macrotemas, ~10–20 s) y F12 (matriz documento × forma de 91k documentos, ~2 min). **Tiempo total en local: ~3 min.**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import figuras as F

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
F.estilo(
    "informe"
)  # "diapositiva" queda disponible (16:9), no se generan esas versiones
D = F.cargar()
print("resultados en", D.R, "→ figuras en", F.dir_figuras())
print("cambios de agenda:", D.agenda_ca)


def salida(fig, fid):
    print(F.guardar(fig, fid))
    plt.show()

## 1. Macrocategorías

El 08 no exportó H, así que la similitud entre tópicos se mide como el **coseno entre columnas de θ** en los documentos de ajuste: dos tópicos se parecen si aparecen en los mismos documentos. Se propone una agrupación **a mano** (columna `macro`, editable en `results/figuras/macrocategorias.csv`) y se compara con el agrupamiento jerárquico de θ (`cluster_theta`, solo sugerencia) mediante el ARI. Si editas el CSV, vuelve a correr desde aquí: los colores siguen el orden de aparición de `macro`.

In [ ]:
MACRO, estado = F.macrocategorias(D)
print("macrocategorias.csv:", estado)
MACRO.sort_values(["macro", "topico"])

## 2. Modelo (F01–F04)

In [ ]:
salida(F.f01_seleccion_k(D), "F01")

In [ ]:
salida(F.f02_bigramas(D), "F02")

In [ ]:
salida(F.f03_topicos(D), "F03")

In [ ]:
salida(F.f04_macrocategorias(D), "F04")

## 3. Agenda y estabilidad (F05–F07)

Serie principal: **El Tiempo sin breves, igual peso por mes válido** (la de control, la más conservadora), con "todos, media por documento" como línea tenue. F06 suma las θ por macrotema y repite el bootstrap del 08 (misma semilla). Se verifica que la suma reproduzca las prevalencias por tópico del 08.

In [ ]:
salida(F.f05_prevalencia_topicos(D), "F05")

In [ ]:
# ⏱ ~10–20 s: bootstrap por macrotema (B del 08)
PM, dif = F.prevalencia_macro(D)
print(f"verificación contra el 08: máx. |Δ prevalencia| = {dif:.2e}")
salida(F.f06_prevalencia_macro(D, PM), "F06")

In [ ]:
salida(F.f07_estabilidad(D), "F07")

## 4. Cambios, artefacto y robustez (F08–F10)

**F08 es la figura clave**: el mismo detector sobre temas y sobre formato, en un eje temporal común, con el régimen de ü del 09 como corroboración independiente. «En el límite» se define con una regla explícita sobre `robustez_cambios.csv` (no robusto, soporte ≥ 50 % y en ≥ 3 de 4 series de control).

In [ ]:
salida(F.f08_prueba_artefacto(D), "F08")

In [ ]:
salida(F.f09_robustez(D), "F09")

In [ ]:
salida(F.f10_linea_tiempo(D), "F10")

## 5. Rareza (F11–F13)

F11a sale de `pmw_zipf.parquet` (N por fuente×año). F11b y F12 se **re-derivan** con el código del 09 y se verifican contra `resumen_09.json`.

In [ ]:
PZ, chk = F.piso_zipf_grupos(D)
print("piso Zipf:", chk)
DEN, ok_den, det_den = F.rederivar_denominador(D)
print("denominador re-derivado coincide con resumen_09:", ok_den, det_den)
salida(F.f11_piso_zipf(D, PZ, DEN if ok_den else None), "F11")
if not ok_den:
    F.INDICE["F11"]["mensaje"] += (
        " (sin panel b: la re-derivación no coincidió; ver results/09/fig_denominador.png)"
    )

In [ ]:
# ⏱ ~2 min: matriz documento × forma (misma que el 09)
AR, ok_ic, det_ic = F.rederivar_ic(D)
print("IC re-derivado coincide con resumen_09:", ok_ic)
print(pd.DataFrame(det_ic).to_string())
if ok_ic:
    salida(F.f12_ic_poisson_bootstrap(D, AR), "F12")
else:
    a, m, f, s = F.FIGURAS["F12"]
    F.INDICE["F12"] = {
        "id": "F12",
        "archivo": "../09/fig_ic_poisson_bootstrap.png",
        "mensaje": m,
        "fuentes_datos": "figura del 09 (la re-derivación no coincidió con resumen_09.json)",
        "seccion_informe": s,
    }
    print(
        "F12 no se exporta: el índice remite a results/09/fig_ic_poisson_bootstrap.png"
    )

In [ ]:
salida(F.f13_zipf_frecuencia_documental(D), "F13")

## 6. Cronolecto (F14–F17)

F17 excluye los términos marcados como `artefacto` en `categorias_lexico.csv` (variantes de ü, tildes, alias/ext).

In [ ]:
salida(F.f14_cuotas_variantes(D), "F14")

In [ ]:
salida(F.f15_paz_lexica(D), "F15")

In [ ]:
salida(F.f16_vecinos_estables(D), "F16")

In [ ]:
salida(F.f17_neologismos(D), "F17")

## 7. Índice

In [ ]:
IDX = F.escribir_indice()
print(F.dir_figuras() / "indice_figuras.csv")
IDX